In [34]:
import os
import pandas as pd
import re
import random
import itertools
import jellyfish
from collections import defaultdict

csv = pd.read_csv("./output.csv", dtype = {"ZipCode": str})
csv.insert(0, "id",[i for i in range(len(csv))])

csv.head()

,id,StreetName,Address,ZipCode,X_COORD,Y_COORD
0,0,BLACK ROCK RD,35,06903,-73.557934,41.178028
1,1,BLACK ROCK RD,79,06903,-73.559713,41.177148
2,2,BLACK ROCK RD,86,06903,-73.559469,41.176574
3,3,BLACK ROCK RD,99,06903,-73.560948,41.176826
4,4,BLACK ROCK RD,111,06903,-73.560947,41.176244


In [35]:
suffix_df = csv.copy()
suffix_df["suffix"] = suffix_df["StreetName"].str.split().str[-1]
suffix_frequencies = suffix_df["suffix"].value_counts()
print(suffix_frequencies.keys())
suffix_df["suffix_freq"] = suffix_df["suffix"].map(suffix_frequencies)
suffix_df = suffix_df.drop_duplicates(subset = ["suffix"])
suffix_df = suffix_df[suffix_df["suffix"].isin(["N","S","E","W","NE","NW","SE","SW"])]
suffix_df = suffix_df.sort_values(by = "suffix_freq", ascending= False)
suffix_df

Index(['RD', 'ST', 'AVE', 'DR', 'LN', 'PL', 'CT', 'CIR', 'TER', 'W', 'TRL',
       'E', 'S', 'BLVD', 'N', 'WAY', 'QUAY', 'PARK', 'TPKE', 'HOLW', 'GRV',
       'DOCK', 'PATH', 'RUN', 'LNDG', 'PLZ', 'EXT', 'WALK', 'PT'],
      dtype='object', name='suffix')


,id,StreetName,Address,ZipCode,X_COORD,Y_COORD,suffix,suffix_freq
2969,2969,DAVENPORT FARM LN W,105,06903,-73.530077,41.130211,W,248
283,283,SPRING HILL LN E,9,06903,-73.553715,41.162960,E,199
479,479,WHITE BIRCH RD S,12,06903,-73.599454,41.159333,S,172
233,233,SPRING HILL LN N,40,06903,-73.557942,41.164729,N,87


In [36]:
prefix_df = csv.copy()
prefix_df["prefix"] = prefix_df["StreetName"].str.split().str[0]
prefix_frequencies = prefix_df["prefix"].value_counts()
print(prefix_frequencies.keys())
prefix_df["prefix_freq"] = prefix_df["prefix"].map(prefix_frequencies)
prefix_df = prefix_df.drop_duplicates(subset = ["prefix"])
prefix_df = prefix_df[prefix_df["prefix"].isin(["N","S","E","W","NE","NW","SE","SW"])]
prefix_df = prefix_df.sort_values(by = "prefix_freq", ascending= False)
prefix_df



Index(['HIGH', 'HOPE', 'W', 'WEST', 'SYLVAN', 'LONG', 'E', 'CONNECTICUT',
       'OLD', 'STILLWATER',
       ...
       'WALTON', 'NOROTON', 'TREGLIA', 'WAMPANAW', 'GARLAND', 'HOWARD',
       'FEDERAL', 'WOODCLIFF', 'HYDE', 'HOLLYWOOD'],
      dtype='object', name='prefix', length=1035)


,id,StreetName,Address,ZipCode,X_COORD,Y_COORD,prefix,prefix_freq
2430,2430,W HAVILAND LN,47,06903,-73.571184,41.136382,W,358
1186,1186,E MIDDLE PATENT RD,375,06831,-73.622433,41.149083,E,260
714,714,N LAKE DR,29,06903,-73.597750,41.155983,N,46
782,782,S LAKE DR,227,06903,-73.608449,41.154805,S,32


In [37]:
## Check for collision where there are any prefixes or suffixes
suffix_and_prefix_df = csv.copy()
suffix_and_prefix_df["suffix"] = suffix_and_prefix_df["StreetName"].str.split().str[-1]
suffix_and_prefix_df["prefix"] = suffix_and_prefix_df["StreetName"].str.split().str[0]
suffix_and_prefix_df = suffix_and_prefix_df[
    suffix_and_prefix_df["suffix"].isin(["N","S","E","W","NE","NW","SE","SW"]) &
    suffix_and_prefix_df["prefix"].isin(["N","S","E","W","NE","NW","SE","SW"])
]

print(suffix_and_prefix_df)
#* Result shows that we don't need to worry about prefix/suffix collision

Empty DataFrame
Columns: [id, StreetName, Address, ZipCode, X_COORD, Y_COORD, suffix, prefix]
Index: []


In [38]:
new_df = csv.copy()

def get_directional(text):
    valid_dirs = ["N","S","E","W"]
    directional = None
    split_txt = text.split()
    directional =  split_txt[0] if split_txt[0] in valid_dirs else directional
    directional = split_txt[-1] if split_txt[-1] in valid_dirs else directional
    return directional
new_df["directional"] = new_df["StreetName"].apply(get_directional)

def drop_directional(text):
    valid_dirs = ["N","S","E","W"]
    split_txt = text.split()
    split_txt = split_txt[1:] if split_txt[0] in valid_dirs else split_txt
    split_txt = split_txt[:-1] if split_txt[-1] in valid_dirs else split_txt
    return " ".join(split_txt)
new_df["StreetName"] = new_df["StreetName"].apply(drop_directional)


def get_ext(text):
    valid_ext = ["EXT"]
    ext = False
    split_txt = text.split()
    ext =  True if split_txt[0] in valid_ext else ext
    ext = True if split_txt[-1] in valid_ext else ext
    return ext
new_df["is_extensional"] = new_df["StreetName"].apply(get_ext)

def drop_ext(text):
    valid_ext = ["EXT"]
    split_txt = text.split()
    split_txt = split_txt[1:] if split_txt[0] in valid_ext else split_txt
    split_txt = split_txt[:-1] if split_txt[-1] in valid_ext else split_txt
    return " ".join(split_txt)
new_df["StreetName"] = new_df["StreetName"].apply(drop_ext)

known_suffixes = ["RD", "ST", "AVE", "DR", "LN", "PL", "CT", "CIR", "TER", "TRL", "BLVD", "WAY", "QUAY", "PARK", "TPKE", "HOLW", "GRV", "DOCK", "PATH", "RUN", "LNDG", "PLZ", "WALK", "PT"]
new_df["street_extension"] = new_df["StreetName"].apply(
    lambda x: x.split()[-1] if x.split()[-1] in known_suffixes else None
)
new_df["StreetName"] = new_df["StreetName"].apply(
    lambda x: " ".join(x.split()[:-1] if x.split()[-1] in known_suffixes else x.split())
) 


new_df
# new_df[~new_df["Directional"].isna()]

,id,StreetName,Address,ZipCode,X_COORD,Y_COORD,directional,is_extensional,street_extension
0,0,BLACK ROCK,35,06903,-73.557934,41.178028,None,False,RD
1,1,BLACK ROCK,79,06903,-73.559713,41.177148,None,False,RD
2,2,BLACK ROCK,86,06903,-73.559469,41.176574,None,False,RD
3,3,BLACK ROCK,99,06903,-73.560948,41.176826,None,False,RD
4,4,BLACK ROCK,111,06903,-73.560947,41.176244,None,False,RD
...,...,...,...,...,...,...,...,...,...
28289,28289,STILLWATER,1019,06902,-73.557396,41.077495,None,False,RD
28290,28290,STILLWATER,1025,06902,-73.557560,41.077722,None,False,RD
28291,28291,STILLWATER,1031,06902,-73.557236,41.077921,None,False,RD
28292,28292,STILLWATER,1041,06902,-73.557844,41.078156,None,False,RD


In [39]:
from pprint import pprint

suffix_abbrev = pd.read_csv("../usps_pub28_street_suffix_abbreviations.csv")

sfx_abrev_lookup = suffix_abbrev.groupby("standard_abbreviation")["commonly_used"].apply(list).to_dict()
# pprint(sfx_abrev_lookup["PL"])

for key, variants in sfx_abrev_lookup.items():
    index = variants.index(key) if key in variants else None
    if index is not  None:
        sfx_abrev_lookup[key] = sfx_abrev_lookup[key][:index] + sfx_abrev_lookup[key][index+1:]
sfx_abrev_lookup = {
    key:variants for key, variants in sfx_abrev_lookup.items() if len(variants) > 0
}

directional_lookup = {
    "N": ["NORTH"],
    "S": ["SOUTH"],
    "E": ["EAST"],
    "W": ["WEST"],
}

In [40]:

def _generate_mask(street_name, pct_flip = 0.1):
    # assert sum(pct_op) == 1 and len(pct_op) == 2
    n = len(street_name)
    mask = [1 if random.random() <= pct_flip else 0 for _ in range(n)]
    while sum(mask) > n//2:
       mask = [1 if random.random() <= pct_flip else 0 for _ in range(n)] 

    #1 = Removal
    #2 = Replacement
    for i, res in enumerate(mask):
        if res == 1:
            if random.random() > 0.5: mask[i] = 2
    return mask

def update_string(street_name, pct2):
    new_street_name = ""
    for i,op in enumerate(_generate_mask(street_name, pct2)):
        c = street_name[i]
        if op == 0:
            new_street_name += street_name[i]
        elif op == 1:
            pass
        elif op == 2:
            if ord('A') <= ord(c) <= ord('Z'):
                new_street_name += random.choice([chr(c) for c in range(ord('A'), ord('Z') + 1)])
            if ord('a') <= ord(c) <= ord('z'):
                new_street_name += random.choice([chr(c) for c in range(ord('a'), ord('z') + 1)])
    return new_street_name


In [41]:
def _corrupt_row(row, pct1=0.7, pct2=0.1, pct3=0.7):

    new_row = row.copy()
    for row_name, lookup in zip(["street_extension", "directional"], [sfx_abrev_lookup, directional_lookup]):
        if row[row_name] in lookup and random.random() < pct1:
            choices = lookup[row[row_name]]
            new_row[row_name] = random.choice(choices)
    new_row["StreetName"] = update_string(row["StreetName"], pct2)
    return new_row

def corrupt_rows(df):
    corrupted_rows = [_corrupt_row(row) for _, row in df.iterrows()]
    new_df = pd.DataFrame(corrupted_rows)
    new_df = new_df.rename(columns = {"id": "original_id"})
    return new_df

corrupted_df = corrupt_rows(new_df)


In [ ]:
#Compiling negative cases

NUM_NEGATIVE = 4

train_df_lst = []
for (StreetName, street_extension, directional), lst in new_df.groupby(["StreetName", "street_extension", "directional"], dropna = False)["id"].apply(list).to_dict().items():
    if len(lst) < 2: continue
    train_df_lst.append(
        pd.Series(
            {
                "StreetName" :StreetName,
                "street_extension": street_extension,
                "directional": directional,
                **{
                    f"id_{i+1}":lst[i] if i < len(lst) else None for i in range(NUM_NEGATIVE) 
                }
            }
        )
    )
train_df = pd.DataFrame(train_df_lst)

jw_lookup = defaultdict(list)
for a,b in itertools.combinations(new_df["StreetName"].unique(), 2):
    score = jellyfish.jaro_winkler_similarity(a, b)
    jw_lookup[a].append((b, score))
    jw_lookup[b].append((a,score))

# print(len(jw_lookup), "names")
# jw_lookup["BLACK ROCK"][:5]

all_scores = [[res, a, b] for a, lst in jw_lookup.items() for [b, res] in lst]
def all_scores_above(thresh = 0.7):
    return [[a,b] for res,a,b in all_scores if res >= thresh]

len(all_scores_above(0.85))


3618

In [44]:
import numpy as np

# distribution of each name's single BEST match score
best_scores = [max(matches, key=lambda x: x[1])[1] for matches in jw_lookup.values()]

print("Best-match score distribution across all names:")
print(f"  min:    {min(best_scores):.3f}")
print(f"  25th %: {np.percentile(best_scores, 25):.3f}")
print(f"  median: {np.percentile(best_scores, 50):.3f}")
print(f"  75th %: {np.percentile(best_scores, 75):.3f}")
print(f"  90th %: {np.percentile(best_scores, 90):.3f}")
print(f"  max:    {max(best_scores):.3f}")
print()

# how many names would have ZERO matches surviving at a few candidate thresholds
for threshold in [0.70, 0.75, 0.80, 0.85, 0.90]:
    n_with_none = sum(1 for s in best_scores if s < threshold)
    print(f"threshold {threshold}: {n_with_none} of {len(best_scores)} names have NO match clearing it")
print()

# eyeball a handful of names' top-5 matches to judge what a given score actually looks like
import random
sample_names = random.sample(list(jw_lookup.keys()), 5)
for name in sample_names:
    top5 = sorted(jw_lookup[name], key=lambda x: -x[1])[:5]
    print(f"{name!r}: {top5}") 

Best-match score distribution across all names:
  min:    0.655
  25th %: 0.822
  median: 0.867
  75th %: 0.900
  90th %: 0.920
  max:    0.982

threshold 0.7: 12 of 1149 names have NO match clearing it
threshold 0.75: 40 of 1149 names have NO match clearing it
threshold 0.8: 191 of 1149 names have NO match clearing it
threshold 0.85: 479 of 1149 names have NO match clearing it
threshold 0.9: 838 of 1149 names have NO match clearing it

'BLUE SPRUCE': [('BLUE ROCK', 0.882828282828283), ('BLUE RIDGE', 0.8672727272727273), ('BEL AIRE', 0.7386363636363638), ('SHELBURNE', 0.6818181818181818), ('BURR', 0.6742424242424242)]
'DOWNS': [('DOLSEN', 0.7911111111111111), ('OWEN', 0.7833333333333333), ('DORSET', 0.76), ('DONATA', 0.76), ('DONALD', 0.76)]
'MELROSE': [('MEADOWS', 0.7714285714285715), ('MERRELL', 0.7714285714285715), ('MERCEDES', 0.7704761904761905), ('SOMERSET', 0.7579365079365079), ('MOORE', 0.7364285714285714)]
'ALDEN': [('ARDEN', 0.88), ('GARDEN', 0.8222222222222223), ('MADELINE',